TF-IDF From Scratch
=====================
In this notebook, we will implement the TF-IDF (Term Frequency-Inverse Document Frequency) algorithm from scratch using Python. TF-IDF is a statistical measure used to evaluate the importance of a word in a document relative to a collection of documents (corpus). It is widely used in information retrieval and text mining.

In [1]:
import math
from collections import defaultdict, Counter
import re
from typing import List, Tuple, Dict



In [2]:
def simple_tokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    return text.split()

In [3]:
class TFIDFRetriever:
    def __init__(self, documents: List[str]):
        self.documents = documents
        self.N = len(documents)

        # 1. Tokenize documents
        self.tokenized_docs = [simple_tokenize(doc) for doc in documents]

        # 2. Build vocab and document frequencies
        self.vocab = set()
        self.df = defaultdict(int)
        self.doc_lengths = []

        for tokens in self.tokenized_docs:
            self.doc_lengths.append(len(tokens))
            seen = set()
            for token in tokens:
                self.vocab.add(token)
                if token not in seen:
                    self.df[token] += 1
                    seen.add(token)
        

        self.vocab = sorted(list(self.vocab))

        
        # 3. Precompute IDF values
        self.idf = {}
        for term in self.vocab:
            self.idf[term] = math.log((self.N + 1) / (self.df[term] + 1)) + 1
        
        # 4. Precompute TF-IDF vectors for documents
        self.doc_vectors = []
        for i, tokens in enumerate(self.tokenized_docs):
            tf_counter = Counter(tokens)
            doc_vector = []
            for term in self.vocab:
                tf = tf_counter[term] / self.doc_lengths[i]
                tfidf = tf * self.idf[term]
                doc_vector.append(tfidf)
            self.doc_vectors.append(doc_vector)

    
    def _vectorize_text(self, text: str) -> Dict[str, float]:
        """Convert input text to TF-IDF vector."""
        tokens = simple_tokenize(text)
        if not tokens:
            return {}
        tf = Counter(tokens)
        total = len(tokens)
        vec = {}
        for term, count in tf.items():
            if term in self.idf:
                tf_val = count / total
                vec[term] = tf_val * self.idf[term]
        return vec

    
    def cosine_similarity(self, vec1: Dict[str, float], vec2: Dict[str, float]) -> float:
        """Compute cosine similarity between two TF-IDF vectors."""
        dot_product = sum(vec1.get(term, 0) * vec2.get(term, 0) for term in self.vocab)
        norm1 = math.sqrt(sum(val ** 2 for val in vec1.values()))
        norm2 = math.sqrt(sum(val ** 2 for val in vec2.values()))
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return dot_product / (norm1 * norm2)

    def retrieve(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Retrieve top_k documents most similar to the query."""
        query_vec = self._vectorize_text(query)
        scores = []
        for i, doc_vec in enumerate(self.doc_vectors):
            doc_vec_dict = {term: doc_vec[idx] for idx, term in enumerate(self.vocab) if doc_vec[idx] > 0}
            score = self.cosine_similarity(query_vec, doc_vec_dict)
            scores.append((i, score))
        
        # Sort by score in descending order and return top_k
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]





    

In [4]:
chunks = [
    "The transformer architecture relies on attention mechanisms to process sequences efficiently.",
    "Backpropagation is the cornerstone of neural network training in deep learning.",
    "Attention is all you need: the paper that introduced transformers.",
    
]
retriever = TFIDFRetriever(chunks)
query = "What is the attention mechanism in transformers?"
results = retriever.retrieve(query, top_k=2)

for idx, score in results:
    print(f"Rank {results.index((idx, score))+1} | Doc {idx} | Score: {score:.4f}")
    print(f"    {chunks[idx]}")

Rank 1 | Doc 2 | Score: 0.4589
    Attention is all you need: the paper that introduced transformers.
Rank 2 | Doc 1 | Score: 0.3267
    Backpropagation is the cornerstone of neural network training in deep learning.


TFIDFVectorizer from SKlearn
-------------------------
After implementing TF-IDF from scratch, we will compare our results with the `TfidfVectorizer` from the `sklearn` library to ensure correctness.


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
corpus = chunks
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(corpus)
print("\nSKlearn TF-IDF Vectors:")
print(X.toarray())

query_vec = vectorizer.transform([query])
similarities = cosine_similarity(query_vec, X).flatten()
top_k_indices = similarities.argsort()[-2:][::-1] # Get top 2 indices

print("\nSKlearn Retrieval Results:")
retrived_chunkes = [corpus[i] for i in top_k_indices]
for rank, idx in enumerate(top_k_indices, start=1):
    print(f"Rank {rank} | Doc {idx} | Score: {similarities[idx]:.4f}")
    print(f"    {corpus[idx]}")







SKlearn TF-IDF Vectors:
[[0.26190578 0.26190578 0.19918609 0.26190578 0.         0.
  0.         0.         0.         0.         0.         0.26190578
  0.         0.         0.         0.26190578 0.26190578 0.
  0.         0.         0.         0.         0.         0.
  0.         0.26190578 0.26190578 0.26190578 0.26190578 0.26190578
  0.26190578 0.         0.         0.26190578 0.26190578 0.        ]
 [0.         0.         0.         0.         0.         0.2773501
  0.2773501  0.2773501  0.2773501  0.2773501  0.2773501  0.
  0.         0.         0.2773501  0.         0.         0.
  0.         0.2773501  0.2773501  0.2773501  0.2773501  0.
  0.         0.         0.         0.         0.         0.
  0.         0.2773501  0.2773501  0.         0.         0.        ]
 [0.         0.         0.25966344 0.         0.34142622 0.
  0.         0.         0.         0.         0.         0.
  0.34142622 0.34142622 0.         0.         0.         0.34142622
  0.34142622 0.         0.

BM25 Implementation
-----------------
We will also implement the BM25 (Best Matching 25) algorithm, which is a ranking function used by search engines to estimate the relevance of documents to a given search query. BM25 is based on the probabilistic retrieval framework and is an improvement over traditional TF-IDF.

In [6]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [simple_tokenize(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
query_tokens = simple_tokenize(query)

bm25_scores = bm25.get_scores(query_tokens)
print("\nBM25 Scores:")
for idx, score in enumerate(bm25_scores):
    print(f"Doc {idx} | Score: {score:.4f}")

docs = bm25.get_top_n(query_tokens, tokenized_corpus, n=2)
print("\nBM25 Retrieval Results:")
for rank, doc in enumerate(docs, start=1):
    print(f"Rank {rank} | Doc: {' '.join(doc)}")



BM25 Scores:
Doc 0 | Score: 0.1726
Doc 1 | Score: 0.6764
Doc 2 | Score: 0.7958

BM25 Retrieval Results:
Rank 1 | Doc: attention is all you need the paper that introduced transformers
Rank 2 | Doc: backpropagation is the cornerstone of neural network training in deep learning
